# PICK: Semi-Supervised Multi-Class Segmentation (BHSD Fold 0)

Phân đoạn 5 phân loại xuất huyết não chi tiết (EDH, IPH, IVH, SAH, SDH) + Background trên GPU Kaggle.

### Cấu hình Kaggle Notebook:
- **Accelerator**: Chọn **GPU T4 x 2** hoặc **GPU P100** (16GB VRAM).
- **Internet**: Bật **ON**.
- **Dataset**: Bấm **+ Add Input** -> Chọn dataset `BHSD Fold 0` đã upload.

### Bước 1: Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q medpy tensorboardX h5py SimpleITK nibabel tensorboard

### Bước 2: Clone & Cập nhật mã nguồn PICK từ GitHub

In [ ]:
import os

REPO_NAME = "NguyenVanCongThanh/PICK_BHSD"
GITHUB_TOKEN = ""  # Điền token ghp_xxx nếu repo là Private, nếu Public thì để trống

if GITHUB_TOKEN.strip():
    GITHUB_REPO_URL = f"https://{GITHUB_TOKEN.strip()}@github.com/{REPO_NAME}.git"
else:
    GITHUB_REPO_URL = f"https://github.com/{REPO_NAME}.git"

WORK_DIR = "/kaggle/working/PICK"

if not os.path.exists(WORK_DIR):
    print(f"[*] Đang clone repository {REPO_NAME}...")
    !git clone {GITHUB_REPO_URL} {WORK_DIR}
else:
    print(f"[OK] Cập nhật git pull tại {WORK_DIR}...")
    !git -C {WORK_DIR} pull

%cd {WORK_DIR}
!git log -1 --oneline

### Bước 3: Định vị Dataset BHSD Fold 0 từ Kaggle Input

In [ ]:
import os, glob

INPUT_DIR = "/kaggle/input"
WORKING_DIR = "/kaggle/working"

def find_bhsd_root(base_dir):
    if not os.path.exists(base_dir):
        return None
    for root, dirs, files in os.walk(base_dir):
        if "data" in dirs and ("splits" in dirs or "fold0_train.txt" in files):
            return root
    return None

print("[*] Đang định vị dataset BHSD Fold 0...")
DATA_ROOT = find_bhsd_root(INPUT_DIR) or find_bhsd_root(WORKING_DIR)

if not DATA_ROOT:
    all_tars = glob.glob(os.path.join(INPUT_DIR, "**", "*.tar"), recursive=True)
    matching_tars = [p for p in all_tars if "bhsd" in os.path.basename(p).lower() or "fold0" in os.path.basename(p).lower()]
    tar_path = matching_tars[0] if matching_tars else (all_tars[0] if all_tars else None)
    
    if not tar_path:
        raise FileNotFoundError(f"[X] Không tìm thấy dataset BHSD Fold 0 trong {INPUT_DIR}! Hãy bấm '+ Add Input'.")
    
    print(f"[*] Đang giải nén {os.path.basename(tar_path)} vào {WORKING_DIR} (khoảng 1-2 phút)...")
    !tar -xf "{tar_path}" -C {WORKING_DIR}
    DATA_ROOT = find_bhsd_root(WORKING_DIR)
    assert DATA_ROOT, f"[X] Giải nén xong nhưng không tìm thấy data/ và splits/ trong {WORKING_DIR}!"

print(f"[SUCCESS] Đường dẫn DATA_ROOT sẵn sàng: {DATA_ROOT}")

### Bước 4: Kiểm tra GPU & Tính toàn vẹn Dữ liệu

In [ ]:
import torch, os, glob

print("=" * 60)
assert torch.cuda.is_available(), "[CẢNH BÁO] Chưa bật GPU! Hãy chọn GPU T4 hoặc P100 ở Notebook Settings."
print(f"[GPU] Thiết bị: {torch.cuda.get_device_name(0)} | VRAM: {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")

train_split = os.path.join(DATA_ROOT, "splits", "fold0_train.txt")
val_split = os.path.join(DATA_ROOT, "splits", "fold0_val.txt")
labeled_split = os.path.join(DATA_ROOT, "train_labeled.txt")
if not os.path.exists(labeled_split):
    labeled_split = os.path.join(DATA_ROOT, "splits", "train_labeled.txt")
data_dir = os.path.join(DATA_ROOT, "data")

for p, name in [(train_split, "fold0_train.txt"), (val_split, "fold0_val.txt"), (data_dir, "data directory")]:
    assert os.path.exists(p), f"[LỖI] Thiếu {name} tại {p}"

with open(train_split) as f: train_cases = [line.strip() for line in f if line.strip()]
with open(val_split) as f: val_cases = [line.strip() for line in f if line.strip()]
labeled_count = len([line.strip() for line in open(labeled_split) if line.strip()]) if os.path.exists(labeled_split) else 0
h5_count = len(glob.glob(os.path.join(data_dir, "*.h5")))

print(f"[+] Số mẫu train:       {len(train_cases)} (Kỳ vọng: 2132)")
print(f"[+] Số mẫu val:         {len(val_cases)} (Kỳ vọng: 38)")
print(f"[+] Số mẫu có nhãn:     {labeled_count} (Kỳ vọng: 154)")
print(f"[+] Tổng file .h5 data: {h5_count} (Kỳ vọng: 2170)")
print("[OK] DỮ LIỆU ĐÃ ĐẦY ĐỦ VÀ SẴN SÀNG HUẤN LUYỆN!")
print("=" * 60)

### Bước 5: (Khuyên dùng) Khởi chạy TensorBoard theo dõi trực quan Real-time
Chạy ô này **TRƯỚC** khi bấm chạy cell huấn luyện (Bước 6). Giao diện TensorBoard sẽ được nhúng trực tiếp ngay trong Notebook:
- **Theo dõi biểu đồ Loss**: Loss tổng thể, Main loss, MIM loss, Aux loss, Learning rate.
- **Theo dõi độ đo Dice**: mDice kiểm định và Dice chi tiết từng phân loại (EDH, IPH, IVH, SAH, SDH) cập nhật mỗi 200 bước.
- **Mẹo**: Bấm biểu tượng bánh răng ⚙️ ở góc trên bên phải TensorBoard và tích chọn **Reload data** (mỗi 30s) để tự động làm mới đồ thị.

In [ ]:
import os
os.makedirs("/kaggle/working/experiments", exist_ok=True)

%reload_ext tensorboard
%tensorboard --logdir /kaggle/working/experiments --port 6006

### Bước 6: Huấn luyện PICK Multi-Class Segmentation (6 classes - Fold 0)
- **Nhiệm vụ**: Phân đoạn 5 phân loại tổn thương xuất huyết não chi tiết (EDH, IPH, IVH, SAH, SDH) + Background.
- **Cơ chế Auto-Resume**: Nếu session bị gián đoạn, chỉ cần chạy lại cell này, script sẽ tự động tìm checkpoint gần nhất (`checkpoint_latest.pth`) để chạy tiếp tục mà không mất bước train nào.
- **Thời gian chạy**: ~60-80 phút trên GPU Kaggle.

In [ ]:
# === HUẤN LUYỆN MULTI-CLASS (FOLD 0) ===
FOLD_ID = 0
EXP_NAME = "BHSD_multiclass_PICK_fold0"
EXP_DIR = "/kaggle/working/experiments"

!python -u BHSD_multiclass_PICK_train.py \
    --root_path {DATA_ROOT} \
    --exp {EXP_NAME} \
    --exp_dir {EXP_DIR} \
    --fold {FOLD_ID} \
    --model VNet \
    --labelnum 154 \
    --max_samples 2132 \
    --batch_size 4 \
    --labeled_bs 2 \
    --pre_max_iteration 4000 \
    --self_max_iteration 12000 \
    --gpu 0 \
    --resume

### Bước 7: Nén và Xuất Kết quả Model Checkpoints & Logs
Nén toàn bộ kết quả (checkpoint tốt nhất `VNet_best_model.pth`, log TensorBoard, và log `train.log`) thành file `.tar.gz` để tải về tại mục **Output**.

In [ ]:
import os

OUTPUT_ARCHIVE = "/kaggle/working/bhsd_multiclass_fold0_results.tar.gz"
EXP_DIR = "/kaggle/working/experiments"

if os.path.exists(EXP_DIR):
    print("[*] Đang nén toàn bộ kết quả mô hình Multi-class và logs...")
    !tar -czvf {OUTPUT_ARCHIVE} -C {EXP_DIR} .
    size_mb = round(os.path.getsize(OUTPUT_ARCHIVE) / (1024**2), 2)
    print(f"\n[OK] Đã nén thành công file kết quả: {OUTPUT_ARCHIVE} ({size_mb} MB)")
    print("[*] File đã sẵn sàng tải về tại tab Output bên phải màn hình!")
else:
    print(f"[!] Chưa tìm thấy thư mục {EXP_DIR}. Vui lòng kiểm tra lại quá trình huấn luyện.")